
# Construct Multi-Omics Unified Gene Matrices

Construct unified multi-omics matrices with:

- Feature scores
- False Discovery Rate (FDR)
- Significance masks

---

## Mathematical Objective

We define a unified gene set:

$$
(1) \quad G^* = G^{MF} \cap G^{GE} \cap G^{CNA} \cap G^{miRNA}
$$

Each omics modality is represented as:

$$
(2) \quad X^{(m)} \in \mathbb{R}^{|G^*| 	imes C}
$$

Final representation:

$$
(3) \quad X^{full} = [X^{score} \mid X^{fdr} \mid X^{sig}]
$$



## Step 1: Import Libraries


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x105fd6790>>
Traceback (most recent call last):
  File "/Users/ericsali/miniforge3/envs/kg39/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 



## Step 2: Load Feature and FDR Matrices

Each omics provides:

- Feature matrix (biological signal)
- FDR matrix (statistical significance)


In [2]:

BASE = Path("../process/fdr/")

mf_feat = pd.read_csv(BASE / "mutation_features_full.csv", index_col=0)
ge_feat = pd.read_csv(BASE / "expression_scores_per_cancer.csv", index_col=0)
cna_feat = pd.read_csv(BASE / "cna_features.csv", index_col=0)
mirna_feat = pd.read_csv(BASE / "gene_from_mirna_features.csv", index_col=0)

mf_fdr = pd.read_csv(BASE / "mutation_fdr_full.csv", index_col=0)
ge_fdr = pd.read_csv(BASE / "expression_fdr_per_cancer.csv", index_col=0)
cna_fdr = pd.read_csv(BASE / "cna_fdr.csv", index_col=0)
mirna_fdr = pd.read_csv(BASE / "gene_from_mirna_fdr.csv", index_col=0)



## Step 3: Remove Duplicate Genes

If duplicate gene entries exist:

$$
(4) \quad x_g = 	ext{median}(x_{g,1}, x_{g,2}, \dots)
$$


In [3]:

def collapse_duplicates(df):
    if df.index.is_unique:
        return df
    return df.groupby(df.index).median()

mf_feat = collapse_duplicates(mf_feat)
ge_feat = collapse_duplicates(ge_feat)
cna_feat = collapse_duplicates(cna_feat)
mirna_feat = collapse_duplicates(mirna_feat)

mf_fdr = collapse_duplicates(mf_fdr)
ge_fdr = collapse_duplicates(ge_fdr)
cna_fdr = collapse_duplicates(cna_fdr)
mirna_fdr = collapse_duplicates(mirna_fdr)



## Step 4: Compute Gene Intersection

We enforce consistency across all omics:

$$
(5) \quad G^* = G^{MF} \cap G^{GE} \cap G^{CNA} \cap G^{miRNA}
$$


In [4]:

genes = sorted(
    set(mf_feat.index)
    & set(ge_feat.index)
    & set(cna_feat.index)
    & set(mirna_feat.index)
)

print("Genes:", len(genes))


Genes: 11160



## Step 5: Align Matrices

Missing values are filled as:

$$
(6) \quad x = 
\begin{cases}
0 & \text{for features} \\
1 & \text{for FDR}
\end{cases}
$$


In [5]:

def align(df, genes, fill=0.0):
    return df.reindex(genes).fillna(fill)

mf_feat = align(mf_feat, genes)
ge_feat = align(ge_feat, genes)
cna_feat = align(cna_feat, genes)
mirna_feat = align(mirna_feat, genes)

mf_fdr = align(mf_fdr, genes, 1.0)
ge_fdr = align(ge_fdr, genes, 1.0)
cna_fdr = align(cna_fdr, genes, 1.0)
mirna_fdr = align(mirna_fdr, genes, 1.0)



## Step 6: Define Significance

We define:

$$
(7) \quad S = \mathbb{1}(\text{FDR} < 0.05)
$$


In [6]:

FDR_THRESH = 0.05

mf_sig = (mf_fdr < FDR_THRESH).astype(int)
ge_sig = (ge_fdr < FDR_THRESH).astype(int)
cna_sig = (cna_fdr < FDR_THRESH).astype(int)
mirna_sig = (mirna_fdr < FDR_THRESH).astype(int)



## Step 7: Build Final Matrices

Each cancer is represented as:

$$
(8) \quad [\text{score}_c, \text{fdr}_c, \text{sig}_c]
$$


In [7]:

def clean(c):
    return c.replace("TCGA-", "")

def build(feat, fdr, sig):
    feat.columns = [clean(c) for c in feat.columns]
    fdr.columns = [clean(c) for c in fdr.columns]
    sig.columns = [clean(c) for c in sig.columns]

    feat = feat.rename(columns=lambda c: f"score_{c}")
    fdr = fdr.rename(columns=lambda c: f"fdr_{c}")
    sig = sig.rename(columns=lambda c: f"sig_{c}")

    return pd.concat([feat, fdr, sig], axis=1)

MF = build(mf_feat, mf_fdr, mf_sig)
GE = build(ge_feat, ge_fdr, ge_sig)
CNA = build(cna_feat, cna_fdr, cna_sig)
miRNA = build(mirna_feat, mirna_fdr, mirna_sig)


In [8]:
MF_score = mf_feat
MF_fdr   = mf_fdr
MF_sig   = (mf_fdr < 0.05).astype(int)

CNA_score = cna_feat
CNA_fdr   = cna_fdr
CNA_sig   = (cna_fdr < 0.05).astype(int)

GE_score = ge_feat
GE_fdr   = ge_fdr
GE_sig   = (ge_fdr < 0.05).astype(int)

miRNA_score = mirna_feat
miRNA_fdr   = mirna_fdr
miRNA_sig   = (mirna_fdr < 0.05).astype(int)



In [9]:
def interleave_columns(score_df, fdr_df, sig_df):
    cancers = score_df.columns

    cols = []
    for c in cancers:
        cols.append(score_df[c].rename(f"score_{c}"))
        cols.append(fdr_df[c].rename(f"fdr_{c}"))
        cols.append(sig_df[c].rename(f"significance_{c}"))
# 
    return pd.concat(cols, axis=1)

MF_final = interleave_columns(MF_score, MF_fdr, MF_sig)
GE_final = interleave_columns(GE_score, GE_fdr, GE_sig)
CNA_final = interleave_columns(CNA_score, CNA_fdr, CNA_sig)
miRNA_final = interleave_columns(miRNA_score, miRNA_fdr, miRNA_sig)

In [10]:
MF_final.index.name = "gene"
GE_final.index.name = "gene"
CNA_final.index.name = "gene"
miRNA_final.index.name = "gene"

# MF_final.to_csv(BASE / "mutation_final.csv")
GE_final.to_csv(BASE / "expression_final.csv")
CNA_final.to_csv(BASE / "cna_final.csv")
miRNA_final.to_csv(BASE / "mirna_final.csv")

# score_df.columns = [c.replace("TCGA-", "") for c in score_df.columns]
# fdr_df.columns = [c.replace("TCGA-", "") for c in fdr_df.columns]
# sig_df.columns = [c.replace("TCGA-", "") for c in sig_df.columns]


## Step 8: Save Outputs


In [ ]:

OUT = Path("data/processed")
OUT.mkdir(exist_ok=True)

for name, df in zip(["MF","GE","CNA","miRNA"], [MF,GE,CNA,miRNA]):
    df.index.name = "gene"
    df.to_csv(OUT / f"{name}_full_matrix.csv")

X = pd.concat([MF, GE, CNA, miRNA], axis=1)
X.index.name = "gene"
X.to_csv(OUT / "X_full_matrix.csv")

print("Final shape:", X.shape)


In [ ]:
def interleave_columns(score_df, fdr_df, sig_df):
    # -------------------------------
    # 1. Align columns strictly
    # -------------------------------
    common = (
        set(score_df.columns)
        & set(fdr_df.columns)
        & set(sig_df.columns)
    )

    common = sorted(common)

    score_df = score_df[common]
    fdr_df   = fdr_df[common]
    sig_df   = sig_df[common]

    # -------------------------------
    # 2. Interleave per cancer
    # -------------------------------
    cols = []

    for c in common:
        cols.append(score_df[c].rename(f"score_{c}"))
        cols.append(fdr_df[c].rename(f"fdr_{c}"))
        cols.append(sig_df[c].rename(f"sig_{c}"))

    return pd.concat(cols, axis=1)

In [ ]:
# -------------------------------
# Build sig again (safe)
# -------------------------------
MF_sig = (mf_fdr < 0.05).astype(int)
GE_sig = (ge_fdr < 0.05).astype(int)
CNA_sig = (cna_fdr < 0.05).astype(int)
miRNA_sig = (mirna_fdr < 0.05).astype(int)

# -------------------------------
# Interleave
# -------------------------------
MF_final = interleave_columns(mf_feat, mf_fdr, MF_sig)
GE_final = interleave_columns(ge_feat, ge_fdr, GE_sig)
CNA_final = interleave_columns(cna_feat, cna_fdr, CNA_sig)
miRNA_final = interleave_columns(mirna_feat, mirna_fdr, miRNA_sig)

# -------------------------------
# Set index name
# -------------------------------
for df in [MF_final, GE_final, CNA_final, miRNA_final]:
    df.index.name = "gene"

# -------------------------------
# Save
# -------------------------------
MF_final.to_csv(BASE / "mutation_final.csv")
GE_final.to_csv(BASE / "expression_final.csv")
CNA_final.to_csv(BASE / "cna_final.csv")
miRNA_final.to_csv(BASE / "mirna_final.csv")